### 1、Import libraries

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.lora import inject_lora
from src.common import get_device

device = get_device()
print(f"The device currently in use is: {device}")

The device currently in use is: cpu


### 2、Load the model (TinyLlama)

In [10]:
import os
os.environ["HF_HOME"] = "D:/AI_Models/huggingface_cache"
#  Define model
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Model downloading.../TinyLlama is being loaded, it may take a few minutes...")
model = AutoModelForCausalLM.from_pretrained(model_id)

model.to(device)
print("Model loaded!")

Model downloading.../TinyLlama is being loaded, it may take a few minutes...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8151.21it/s]


Model loaded!


### 3、Check the model before injecting Lora (mainly focusing on the number of parameters, which is approximately 1.1 billion for TinyLlama)

In [11]:
def print_trainable_parameters(model):
    """
    The number of parameters that need to be trained and the total number of parameters
    """
    trainable_params = 0
    all_param = 0
    
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
            
    print(f"Total number of parameters: {all_param:,}")
    print(f"The number of parameters that need to be trained: {trainable_params:,}")
    print(f"Proportion: {100 * trainable_params / all_param:.4f}%\n")

print("Check the model BEFORE injecting the Lora")
print_trainable_parameters(model)

Check the model BEFORE injecting the Lora
Total number of parameters: 1,100,048,384
The number of parameters that need to be trained: 1,100,048,384
Proportion: 100.0000%



### 4、Inject Lora (freeze the parameters of TinyLlama, only fine-tune the Lora and the proportion of parameters has decreased to about 0.1%.)

In [12]:
print("Start to inject Lora...")

for param in model.parameters():
    param.requires_grad = False

model = inject_lora(model, target_modules=("q_proj", "v_proj"), r=8, alpha=16)

print("Lora injected! Check the model AFTER injecting the Lora")
print_trainable_parameters(model)

Start to inject Lora...
Lora injected! Check the model AFTER injecting the Lora
Total number of parameters: 1,101,174,784
The number of parameters that need to be trained: 1,126,400
Proportion: 0.1023%

